# CVAE Counterfactual Generation

This notebook demonstrates counterfactual generation using the AEGIS
Conditional Variational Autoencoder (CVAE).

**Goals:**
1. Load and prepare data
2. Train the CVAE
3. Generate counterfactuals
4. Visualize the latent space
5. Validate counterfactuals
6. Summary

In [ ]:
# Cell 1: Imports
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import time

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

try:
    import torch
    import torch.nn as nn
    print(f"PyTorch: {torch.__version__}")
    HAS_TORCH = True
except ImportError:
    print("PyTorch: not installed")
    HAS_TORCH = False

import matplotlib.pyplot as plt
print(f"Matplotlib available: True")

In [ ]:
# Cell 2: Load and prepare data
n_samples = 1000
n_features = 10

X_raw, y = make_classification(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=7,
    n_redundant=2,
    n_classes=2,
    random_state=42,
)

# Create sensitive attribute (synthetic)
sensitive = (X_raw[:, 0] > np.median(X_raw[:, 0])).astype(np.float32)

# Normalize features to [0, 1]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Scale to [0, 1] for VAE
col_mins = X_scaled.min(axis=0)
col_maxs = X_scaled.max(axis=0)
col_range = col_maxs - col_mins
col_range[col_range < 1e-8] = 1.0
X_normalized = (X_scaled - col_mins) / col_range
X_normalized = np.clip(X_normalized, 0.0, 1.0).astype(np.float32)

X_train, X_val, y_train, y_val, s_train, s_val = train_test_split(
    X_normalized, y, sensitive, test_size=0.2, random_state=42, stratify=y,
)

# Condition is the sensitive attribute (single dim)
C_train = s_train.reshape(-1, 1)
C_val = s_val.reshape(-1, 1)

print(f"Train: {X_train.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")
print(f"Condition dim: {C_train.shape[1]}")
print(f"Sensitive distribution (train): group 0={sum(s_train==0)}, group 1={sum(s_train==1)}")

feature_names = [f"feature_{i}" for i in range(n_features)]

In [ ]:
# Cell 3: Train CVAE
if HAS_TORCH:
    from app.ml.neural.conditional_vae import ConditionalVAE
    from app.ml.neural.vae_trainer import VAETrainer, VAETrainingConfig
    
    # Create model
    cvae = ConditionalVAE(
        input_dim=n_features,
        condition_dim=1,
        latent_dim=16,
        dropout=0.1,
        output_activation='sigmoid',
    )
    
    print(f"CVAE created: {sum(p.numel() for p in cvae.parameters())} parameters")
    
    # Training config
    config = VAETrainingConfig(
        epochs=50,            # Reduced for demo
        batch_size=64,
        learning_rate=1e-3,
        kl_annealing_epochs=10,
        early_stopping_patience=10,
        checkpoint_dir='/tmp/cvae_checkpoints',
        save_every=10,
    )
    
    # Train
    trainer = VAETrainer(cvae, config=config)
    
    print("\nStarting CVAE training...")
    t0 = time.time()
    history = trainer.train(
        train_data=X_train,
        condition_data=C_train,
        val_data=X_val,
        val_condition=C_val,
    )
    elapsed = time.time() - t0
    
    print(f"\nTraining complete in {elapsed:.2f}s")
    print(f"  Best epoch:      {history.best_epoch}")
    print(f"  Best val loss:   {history.best_val_loss:.6f}")
    print(f"  Final train loss: {history.train_losses[-1]:.6f}")
    print(f"  Final KL loss:   {history.kl_losses[-1]:.6f}")
else:
    print("PyTorch not available. Skipping CVAE training.")

In [ ]:
# Cell 4: Generate counterfactuals
if HAS_TORCH:
    from app.ml.neural.counterfactual_generator import CounterfactualGenerator
    
    generator = CounterfactualGenerator(
        cvae=cvae,
        feature_names=feature_names,
        device='cpu',
    )
    
    # Generate counterfactuals for 5 samples
    print("Generating counterfactuals...")
    print("(Changing sensitive attribute from group 0 to group 1)\n")
    
    cf_results = []
    n_cf = min(5, len(X_val))
    
    for i in range(n_cf):
        result = generator.generate_counterfactual(
            original_sample=X_val[i],
            sensitive_attribute_idx=0,
            original_value=float(s_val[i]),
            target_value=1.0 - float(s_val[i]),
            condition_dim=1,
        )
        cf_results.append(result)
    
    # Display results
    for i, cf in enumerate(cf_results):
        print(f"Sample {i}:")
        print(f"  Sensitive: {cf.original_value} -> {cf.target_value}")
        print(f"  Features changed: {len(cf.feature_changes)}")
        for fname, change in sorted(cf.feature_changes.items(), key=lambda x: -abs(x[1]['change'])):
            print(f"    {fname}: {change['original']:.4f} -> {change['counterfactual']:.4f} (delta={change['change']:+.4f})")
        print()
else:
    print("PyTorch not available. Skipping counterfactual generation.")

In [ ]:
# Cell 5: Visualize latent space
if HAS_TORCH:
    cvae.eval()
    
    # Encode all validation samples
    with torch.no_grad():
        x_tensor = torch.FloatTensor(X_val)
        c_tensor = torch.FloatTensor(C_val)
        mu, log_var = cvae.encode(x_tensor, c_tensor)
        z = mu.detach().numpy()
    
    # Plot latent space (first 2 dims)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Color by class
    scatter = axes[0].scatter(z[:, 0], z[:, 1], c=y_val, cmap='viridis', alpha=0.6, s=20)
    axes[0].set_title('Latent Space (colored by class)')
    axes[0].set_xlabel('z[0]')
    axes[0].set_ylabel('z[1]')
    plt.colorbar(scatter, ax=axes[0], label='Class')
    
    # Color by sensitive attribute
    scatter2 = axes[1].scatter(z[:, 0], z[:, 1], c=s_val, cmap='coolwarm', alpha=0.6, s=20)
    axes[1].set_title('Latent Space (colored by sensitive attr)')
    axes[1].set_xlabel('z[0]')
    axes[1].set_ylabel('z[1]')
    plt.colorbar(scatter2, ax=axes[1], label='Sensitive Group')
    
    plt.tight_layout()
    plt.show()
    print("Latent space visualization rendered above.")
    
    # Latent space statistics
    print(f"\nLatent space statistics:")
    print(f"  Mean per dim: {z.mean(axis=0).round(3)}")
    print(f"  Std per dim:  {z.std(axis=0).round(3)}")
else:
    print("PyTorch not available. Skipping latent space visualization.")

In [ ]:
# Cell 6: Validate counterfactuals
if HAS_TORCH and cf_results:
    from sklearn.linear_model import LogisticRegression
    
    # Train a simple classifier on the original data
    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_train, y_train)
    print(f"Classifier accuracy: {clf.score(X_val, y_val):.4f}")
    
    # Validate each counterfactual
    print(f"\nCounterfactual Validation:")
    print(f"{'Sample':>8} {'Orig Pred':>10} {'CF Pred':>10} {'Changed':>10}")
    print("-" * 42)
    
    n_changed = 0
    for i, cf in enumerate(cf_results):
        validation = generator.validate_counterfactual(
            original=cf.original,
            counterfactual=cf.counterfactual,
            model=clf,
        )
        if validation.get('prediction_changed', False):
            n_changed += 1
        orig_pred = validation.get('original_prediction', 'N/A')
        cf_pred = validation.get('counterfactual_prediction', 'N/A')
        changed = 'YES' if validation.get('prediction_changed', False) else 'NO'
        print(f"{i:>8} {str(orig_pred):>10} {str(cf_pred):>10} {changed:>10}")
    
    print(f"\nPrediction changed in {n_changed}/{len(cf_results)} counterfactuals ({n_changed/len(cf_results)*100:.0f}%)")
    
    # Average L2 distance between original and counterfactual
    distances = []
    for cf in cf_results:
        dist = np.linalg.norm(cf.original - cf.counterfactual)
        distances.append(dist)
    
    print(f"\nCounterfactual quality metrics:")
    print(f"  Avg L2 distance:  {np.mean(distances):.4f}")
    print(f"  Min L2 distance:  {np.min(distances):.4f}")
    print(f"  Max L2 distance:  {np.max(distances):.4f}")
    print(f"  Avg features changed: {np.mean([len(cf.feature_changes) for cf in cf_results]):.1f}")
else:
    print("Skipping counterfactual validation.")

In [ ]:
# Cell 7: Summary
print("=" * 60)
print("CVAE Counterfactual Generation - Summary")
print("=" * 60)

if HAS_TORCH:
    print(f"""
Data:
  - {n_samples} samples, {n_features} features
  - Sensitive attribute: synthetic binary
  - Train/Val split: {X_train.shape[0]}/{X_val.shape[0]}

CVAE Training:
  - Latent dim: 16
  - Condition dim: 1 (sensitive attribute)
  - Best epoch: {history.best_epoch}
  - Best val loss: {history.best_val_loss:.6f}

Counterfactual Generation:
  - Generated {len(cf_results)} counterfactuals
  - Avg features changed per sample: {np.mean([len(cf.feature_changes) for cf in cf_results]):.1f}
  - Prediction changed in {n_changed}/{len(cf_results)} cases ({n_changed/len(cf_results)*100:.0f}%)

Key Takeaways:
  1. The CVAE learns to encode data conditioned on the sensitive attribute.
  2. Counterfactuals show what a sample would look like with a different
     sensitive attribute value.
  3. The latent space separates both by class and by sensitive group.
  4. Counterfactuals can be validated by checking if model predictions change.
""")
else:
    print("\nNo results available (PyTorch required for CVAE).")
    print("Install PyTorch: pip install torch")